In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier
)

from xgboost import XGBClassifier

In [3]:
train = pd.read_csv(
    r"C:\Users\hecto\Documents\PythonProjects\obesity_risk\train.csv"
)

test = pd.read_csv(
    r"C:\Users\hecto\Documents\PythonProjects\obesity_risk\test.csv"
)

train.head()

,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:
#Define Features and Target
target = "NObeyesdad"

X = train.drop(columns=["id", target])

y = train[target]

In [5]:
#Encode Target Variable
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

In [6]:
#Identify Variable Types
categorical_cols = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    exclude=["object", "string"]
).columns.tolist()

print("Categorical Columns:")
print(categorical_cols)

print("\nNumeric Columns:")
print(numeric_cols)

Categorical Columns:
['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']

Numeric Columns:
['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']


In [7]:
#Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

In [8]:
#Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

## Model 1: Decision Tree

In [9]:
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

tree_acc = accuracy_score(y_test, tree_pred)

print("Decision Tree Accuracy:", tree_acc)

print(classification_report(
    y_test,
    tree_pred,
    target_names=label_encoder.classes_
))

Decision Tree Accuracy: 0.8427263969171483
                     precision    recall  f1-score   support

Insufficient_Weight       0.88      0.89      0.89       505
      Normal_Weight       0.80      0.79      0.79       617
     Obesity_Type_I       0.81      0.82      0.81       582
    Obesity_Type_II       0.94      0.94      0.94       650
   Obesity_Type_III       0.99      0.99      0.99       809
 Overweight_Level_I       0.67      0.65      0.66       485
Overweight_Level_II       0.69      0.71      0.70       504

           accuracy                           0.84      4152
          macro avg       0.83      0.83      0.83      4152
       weighted avg       0.84      0.84      0.84      4152



## Decision Tree Interpretation

The Decision Tree classifier achieved approximately 84.3% classification accuracy on the obesity risk dataset. The model performed especially well for the Obesity_Type_II and Obesity_Type_III categories, but classification performance was weaker for the overweight categories where class overlap was more substantial. Decision trees partition the predictor space into rectangular regions through recursive binary splitting, allowing the model to capture nonlinear relationships and interaction effects among predictors. However, single decision trees are prone to overfitting and can be sensitive to small changes in the training data. As a result, while the decision tree provided reasonable predictive performance and strong interpretability, it served primarily as a baseline model before applying more advanced ensemble tree-based methods.

## Model 2: Bagging

In [10]:
bag_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42
    ))
])

bag_model.fit(X_train, y_train)

bag_pred = bag_model.predict(X_test)

bag_acc = accuracy_score(y_test, bag_pred)

print("Bagging Accuracy:", bag_acc)

print(classification_report(
    y_test,
    bag_pred,
    target_names=label_encoder.classes_
))

Bagging Accuracy: 0.8894508670520231
                     precision    recall  f1-score   support

Insufficient_Weight       0.93      0.94      0.93       505
      Normal_Weight       0.86      0.86      0.86       617
     Obesity_Type_I       0.87      0.88      0.87       582
    Obesity_Type_II       0.96      0.97      0.97       650
   Obesity_Type_III       1.00      0.99      0.99       809
 Overweight_Level_I       0.76      0.72      0.74       485
Overweight_Level_II       0.77      0.78      0.78       504

           accuracy                           0.89      4152
          macro avg       0.88      0.88      0.88      4152
       weighted avg       0.89      0.89      0.89      4152



## Bagging Model Interpretation

The bagging classifier achieved approximately 88.9% classification accuracy, improving substantially over the single decision tree model. Bagging works by fitting multiple decision trees on bootstrap samples of the training data and averaging their predictions to reduce variance and improve stability. The model demonstrated stronger predictive performance across nearly all obesity categories, particularly for Obesity_Type_I, Obesity_Type_II, and the overweight categories. The improvement in accuracy indicates that averaging many trees helped reduce overfitting and allowed the model to better generalize to unseen observations. These results suggest that ensemble learning methods are more effective than a single decision tree for capturing the nonlinear relationships and interaction effects present within the obesity risk dataset.

## Model 3: Random Forest

In [11]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_features="sqrt",
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)

print(classification_report(
    y_test,
    rf_pred,
    target_names=label_encoder.classes_
))

Random Forest Accuracy: 0.8966763005780347
                     precision    recall  f1-score   support

Insufficient_Weight       0.93      0.93      0.93       505
      Normal_Weight       0.84      0.88      0.86       617
     Obesity_Type_I       0.88      0.88      0.88       582
    Obesity_Type_II       0.97      0.97      0.97       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.81      0.72      0.76       485
Overweight_Level_II       0.79      0.81      0.80       504

           accuracy                           0.90      4152
          macro avg       0.89      0.89      0.89      4152
       weighted avg       0.90      0.90      0.90      4152



## Random Forest Interpretation

The Random Forest classifier achieved approximately 89.7% classification accuracy, outperforming both the single decision tree and bagging models. Random forests improve upon bagging by introducing random subsets of predictors at each split, which reduces correlation among the trees and improves model generalization. The model demonstrated strong predictive performance across nearly all obesity categories, particularly for Obesity_Type_II and Obesity_Type_III, while also improving classification performance for the overweight categories. The improved accuracy suggests that random forests effectively captured nonlinear relationships and interaction effects within the obesity risk dataset while simultaneously reducing overfitting. These findings demonstrate the effectiveness of random forests as a robust ensemble learning approach for complex multi-class classification problems.

## Model 4: XGBoost

In [12]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softmax",
        num_class=len(label_encoder.classes_),
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_pred)

print("XGBoost Accuracy:", xgb_acc)

print(classification_report(
    y_test,
    xgb_pred,
    target_names=label_encoder.classes_
))

XGBoost Accuracy: 0.9096820809248555
                     precision    recall  f1-score   support

Insufficient_Weight       0.94      0.95      0.94       505
      Normal_Weight       0.88      0.90      0.89       617
     Obesity_Type_I       0.89      0.90      0.89       582
    Obesity_Type_II       0.97      0.97      0.97       650
   Obesity_Type_III       1.00      1.00      1.00       809
 Overweight_Level_I       0.81      0.78      0.79       485
Overweight_Level_II       0.81      0.81      0.81       504

           accuracy                           0.91      4152
          macro avg       0.90      0.90      0.90      4152
       weighted avg       0.91      0.91      0.91      4152



## XGBoost Interpretation

The XGBoost classifier achieved approximately 91.0% classification accuracy, producing the strongest overall performance among all models evaluated. XGBoost is a boosting algorithm that sequentially builds decision trees where each new tree attempts to correct errors made by previous trees. The model demonstrated excellent precision, recall, and F1-scores across nearly all obesity categories, including perfect classification performance for Obesity_Type_III. Performance also improved substantially for the overweight and moderate obesity categories compared to the previous models.

The superior performance of XGBoost indicates that the obesity risk dataset contains complex nonlinear relationships and interaction effects that are effectively captured through boosted ensemble learning techniques. By combining sequential learning, regularization, and gradient optimization, XGBoost reduced classification errors and improved model generalization more effectively than the decision tree, bagging, and random forest models.

XGBoost was used instead of BART because it provides similar boosting-based ensemble capabilities while offering easier implementation, faster computation, and better integration within the Python machine learning ecosystem.

## Model Comparison

| Model | Accuracy |
|---|---|
| Decision Tree | 0.843 |
| Bagging | 0.889 |
| Random Forest | 0.897 |
| XGBoost | 0.910 |

The results demonstrate that ensemble learning methods substantially outperformed the single decision tree model. Decision trees provided reasonable baseline performance but were more susceptible to overfitting and instability. Bagging improved classification accuracy by averaging predictions from multiple trees, thereby reducing variance. Random forests further improved performance by introducing random feature selection at each split, reducing correlation among trees and improving generalization. XGBoost produced the strongest overall results because boosting sequentially corrected prior model errors while effectively modeling nonlinear relationships and complex predictor interactions.

## Assumption Investigation

The tree-based methods evaluated in this analysis require fewer strict statistical assumptions than traditional linear models. Decision trees, bagging, random forests, and XGBoost do not assume linear relationships between predictors and the response variable, making them well suited for datasets containing nonlinear patterns and interaction effects. All models assume that observations are independent and that the training data is representative of the broader population.

The lower performance of the single decision tree suggests that individual trees may overfit portions of the training data and struggle to generalize complex class boundaries. Bagging and random forests reduced variance and improved generalization by averaging predictions across multiple trees. Random forests additionally reduced correlation among trees through random feature selection. XGBoost provided the strongest predictive performance because boosting iteratively corrected classification errors while incorporating regularization techniques to control overfitting. These findings indicate that the obesity risk dataset contains substantial nonlinear structure and interaction effects that are best modeled through advanced ensemble learning approaches.

## Submission Files

In [13]:
#Decision Tree
tree_test_pred = tree_model.predict(test.drop(columns=["id"]))

tree_labels = label_encoder.inverse_transform(tree_test_pred)

tree_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": tree_labels
})

tree_submission.to_csv(
    "obesity_decision_tree_submission.csv",
    index=False
)

print("Decision Tree submission file created.")

Decision Tree submission file created.


In [14]:
#Bagging Submission
bag_test_pred = bag_model.predict(test.drop(columns=["id"]))

bag_labels = label_encoder.inverse_transform(bag_test_pred)

bag_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": bag_labels
})

bag_submission.to_csv(
    "obesity_bagging_submission.csv",
    index=False
)

print("Bagging submission file created.")

Bagging submission file created.


In [15]:
#Random Forest
rf_test_pred = rf_model.predict(test.drop(columns=["id"]))

rf_labels = label_encoder.inverse_transform(rf_test_pred)

rf_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": rf_labels
})

rf_submission.to_csv(
    "obesity_random_forest_submission.csv",
    index=False
)

print("Random Forest submission file created.")

Random Forest submission file created.


In [16]:
#XGBoost Submission
xgb_test_pred = xgb_model.predict(test.drop(columns=["id"]))

xgb_labels = label_encoder.inverse_transform(xgb_test_pred)

xgb_submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": xgb_labels
})

xgb_submission.to_csv(
    "obesity_xgboost_submission.csv",
    index=False
)

print("XGBoost submission file created.")

XGBoost submission file created.
